# 초구 예측 — model_11 (3단계: 위치 1단계 보완 = 시즌 초 배수)

- 3단계 맨 앞 "위치 1단계 보완" (README 5장 3번, 4-23)
  - 0장 0-1 진단: 퍼짐·보정은 이미 맞음, 투수 고유 문제 아님, **시즌 첫 1–3등판에서 두 해 같은 방향으로 어긋남**, 구종별 위치 차이 기대 > 실제
  - 적용 순서 (사용자 확인): 시즌 초 -> 투수 전체 성향·모든 카운트 중 큰 쪽 -> 남은 쪽 -> 분산 따로 (시즌 초 다음 모델 0장에서 크기 재서 확정)
- 모델: model_09에서 **위치에 시즌 초 배수만** 더함 (구종 model_06 B4와 같은 방식)
  - 새 시즌 첫 경기 전에 지난 시즌까지 투수×구종·투수×구종×손 위치 누적 무게에 배수를 곱함
  - 위치 파라미터(반감기·k_type·k_hand)를 시즌 초 배수와 함께 2019로 다시 고름 (실제 구종 기준 24칸 log loss)
  - 구종(B4 + LightGBM 월 단위 재학습)은 model_09 그대로 -> 실전 변화 = 순수한 위치 개선 효과
- 비교 대상: model_09

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_11"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

- 0-1: 지금 위치 모델(model_09 = model_03 값)이 어디서 어긋나는지 -> 투수 문제인지, 기록 수·시즌 초·퍼짐 문제인지, 상황 문제인지
- 0-2: 1단계 보완 후보 세 가지의 크기를 지금 값 주변에서 재서 순서를 정함 (다음)
- 판단은 2019(검증), 2020은 같은 방향인지 확인만

### 0-1. 위치 적중·퍼짐 진단

- 대상: model_09 위치 (= model_03 값: 기준점 decay 173구, 반감 35.4, k_type 1000, k_hand 17.3, 24칸 공분산 적분)
- 기본은 **실제 구종 기준** P(칸|실제 구종) = 위치 모델 자체 (구종 오류 제외, 위치 파라미터 고르는 기준과 같음), 실전은 0-1-6에서만
- 용어
  - 포함: 누적 80%까지 보여 준 칸 안에 실제 칸이 있음 / 기대 포함률 = 보여 준 칸 확률 합 / gap = 포함 − 기대 (음수 = 생각보다 자주 놓침)
  - llx = 24칸 log loss − 엔트로피 (양수 = 과신, 음수 = 과소 확신)
  - rC = 존 밖(C) 실제(0/1) − 예측 확률 (양수 = C를 덜 예측), rH는 Heart 같은 방식
  - 퍼짐: 표준화 잔차 ex = (실제 좌우 − 예측 중심) / 예측 SD, ez는 높이(cm) 같은 방식
    - 평균 ex·ez ≠ 0 -> 중심 치우침 (SD 단위) / SD(ex)·SD(ez) > 1 -> 예측 퍼짐이 좁음, < 1 -> 넓음
    - 80% 타원: 예측 2차원 정규분포의 80% 타원(마할라노비스 d² ≤ 3.22) 안에 실제 위치가 든 비율 -> 0.80이면 맞음 (50% 타원은 d² ≤ 1.39)
    - 튀는 공에 덜 민감한 SD: MAD × 1.4826 (정규분포면 SD와 같음)
  - 좌우는 타자 기준 (+ = 몸쪽)

#### 0-1-1. 공통 계산

- 위치 함수는 model_09 3장 복사 (값도 같음) -> 실제 구종 기준 24칸 log loss가 model_09 결과 파일과 같은지 확인
- 그룹 변수: 이번 타석 상황 `fc.situation_features` (경기 기록, 그 공 앞), 투수×구종×손 전날까지 초구 수, 그 시즌 몇 번째 등판 (경기 기록)

In [2]:
DIAG = "model_09"
pp13 = fc.load_results([DIAG])[DIAG]  # 구종 확률(실전 섞기용)·재현 확인용
LOC13 = dict(half_life=35.4, k_type=1000.0, k_hand=17.3, ref="decay", ref_hl=173.0)  # model_09 위치 값 (= model_03)
DIAG_ZONE_M = 32  # 24칸 적분 점 수

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 위치 모멘트 + 24칸 비율 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포 (t: 이름 하나 또는 공마다 배열)
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl), qt)  # 모멘트 -> 정규분포 (높이는 그 타자 존으로 cm 환산)
    return fc.zone_from_gauss_cov(g, qt, DIAG_ZONE_M), g  # 정규분포를 24칸 위에서 적분 (공분산 반영)

# 그룹 변수 (모두 그 공 앞 정보)
SIT = fc.situation_features(m, fp)  # 이번 타석 상황 A (fp 행 순서)
_, NH = fc.lookup(fc.prior_table(fp, K2, ["x"]), fp, K2, ["x"])  # 투수×구종×손 전날까지 초구 수 (가중 없음)
app = m.drop_duplicates(["pitcher_code", "gameID"])[["pitcher_code", "gameID", "game_date", "GYEAR"]].sort_values(["game_date", "gameID"])
app["app_no"] = app.groupby(["pitcher_code", "GYEAR"]).cumcount() + 1  # 그 시즌 몇 번째 등판 (경기 기록, 41명 공이 있는 경기만)
APP_NO = fp[["pitcher_code", "gameID"]].merge(app[["pitcher_code", "gameID", "app_no"]], how="left", validate="many_to_one")["app_no"].to_numpy()
GRP = fc.ZONE_GROUP  # 24칸 -> H/S/C

def prep(year):  # 그 해 초구 공별 진단 값 (fp 순서 = fc.align 순서)
    sel = fp["GYEAR"].eq(year).to_numpy()
    q = fp[sel]
    i = np.arange(len(q))
    yz, yt = q["zone"].map(fc.ZONE_IDX).to_numpy(), q["pitch_type"].map(fc.TYPE_IDX).to_numpy()  # 실제 칸·구종 번호
    PZ, g = loc_given_type(q, year, q["pitch_type"].to_numpy(), **LOC13)  # 실제 구종 기준 24칸 확률 (n, 24), 정규분포
    mx, mz, vx, vz, c = g  # 중심 (좌우, 높이 cm), 분산, 공분산
    dx, dz = q["x"].to_numpy() - mx, q["CROSS_PLATE_Z_CM"].to_numpy() - mz  # 실제 − 예측 중심
    d2 = (vz * dx ** 2 - 2 * c * dx * dz + vx * dz ** 2) / (vx * vz - c ** 2)  # 마할라노비스 거리 제곱 (정규분포면 자유도 2 카이제곱)
    lp = np.log(np.clip(PZ, 1e-12, None))
    show = fc.show_types(PZ)  # 누적 80%까지 보여 줄 칸 (n, 24)
    d = q[fc.KEYS + ["pitcher_code", "pitch_type", "same_side", "game_date", "zone", "zone_group"]].copy()
    d["ll"] = -lp[i, yz]  # 24칸 log loss (실제 구종)
    d["llx"] = d["ll"] + (PZ * lp).sum(1)  # log loss − 엔트로피
    d["hit"] = show[i, yz].astype(float)
    d["exp"] = (PZ * show).sum(1)  # 기대 포함률
    d["gap"] = d["hit"] - d["exp"]
    d["pC"] = PZ[:, GRP == "C"].sum(1)  # 존 밖 예측 확률
    d["rC"] = q["zone_group"].eq("C").to_numpy() - d["pC"]
    d["rH"] = q["zone_group"].eq("H").to_numpy() - PZ[:, GRP == "H"].sum(1)
    d["ex"], d["ez"] = dx / np.sqrt(vx), dz / np.sqrt(vz)  # 표준화 잔차 (축별)
    d["in80"], d["in50"] = (d2 <= -2 * np.log(0.2)).astype(float), (d2 <= 2 * np.log(2)).astype(float)  # 80%·50% 타원 안
    d["sx"], d["sz"] = np.sqrt(vx), np.sqrt(vz)  # 예측 SD cm
    # 실전: model_09 구종 확률로 섞기
    Pt = fc.align(pp13, fp, year)[fc.P_TYPE].to_numpy()  # (n, 9), fp 순서
    PZall = np.stack([loc_given_type(q, year, t, **LOC13)[0] for t in CLASSES], 1)  # 구종 9개 각각 가정 (n, 9, 24)
    Pmix = np.einsum("nt,ntz->nz", Pt, PZall)  # 실전 24칸 확률
    d["ll_mix"] = -np.log(np.clip(Pmix[i, yz], 1e-12, None))
    d["kl"] = (Pt * (PZall * (np.log(np.clip(PZall, 1e-12, None)) - np.log(np.clip(Pmix, 1e-12, None))[:, None, :])).sum(2)).sum(1)  # 모델이 기대한 몫 (구종과 칸의 상호정보)
    d["pC_mix"] = Pmix[:, GRP == "C"].sum(1)
    d["p_true_type"] = Pt[i, yt]  # 실제 구종에 준 확률
    # 그룹 변수
    d[fc.A_COLS] = SIT[sel].to_numpy()
    d["n_hand"] = NH[sel, 0]
    d["app_no"] = APP_NO[sel]
    # 재현 확인: model_09 결과 파일과 같은지
    chk = fc.align(pp13, fp, year)
    assert np.allclose(d["ll"].to_numpy(), chk["ll_zone_true"].to_numpy()), "실제 구종 기준 24칸 log loss가 model_09과 다름"
    assert np.allclose(d["ll_mix"].to_numpy(), -np.log(np.clip(chk[fc.P_ZONE].to_numpy()[i, yz], 1e-12, None))), "실전 24칸이 model_09과 다름"
    return d.reset_index(drop=True)

D = {yr: prep(yr) for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}
print("실제 구종 기준·실전 24칸 log loss = model_09 (재현 확인)")

실제 구종 기준·실전 24칸 log loss = model_09 (재현 확인)


#### 0-1-2. 전체 보정·퍼짐

- 해 전체 평균 (아래 표의 기준), 구종별 표는 2019

In [3]:
mad_sd = lambda s: 1.4826 * np.median(np.abs(s - np.median(s)))  # 튀는 공에 덜 민감한 SD

def summ(d):  # 한 묶음의 보정·퍼짐 요약
    return pd.Series({"공": len(d), "24칸 ll": d.ll.mean(), "llx": d.llx.mean(), "포함률": d.hit.mean(), "기대 포함률": d.exp.mean(),
                      "C 예측": d.pC.mean(), "C 실제": (d.rC + d.pC).mean(), "rC": d.rC.mean(), "rH": d.rH.mean(),
                      "평균 ex": d.ex.mean(), "평균 ez": d.ez.mean(), "SD ex": d.ex.std(), "SD ez": d.ez.std(),
                      "MAD SD ex": mad_sd(d.ex), "MAD SD ez": mad_sd(d.ez), "50% 타원": d.in50.mean(), "80% 타원": d.in80.mean(),
                      "예측 SD 좌우 cm": d.sx.mean(), "예측 SD 높이 cm": d.sz.mean()})

display(pd.DataFrame({yr: summ(D[yr]) for yr in D}).round(4))
print("구종별 (2019)")
d = D[fc.VAL_YEAR]
display(pd.DataFrame({t: summ(d[d.pitch_type.eq(t)]) for t in CLASSES if d.pitch_type.eq(t).sum() >= 30}).T
        [["공", "llx", "rC", "평균 ex", "평균 ez", "SD ex", "SD ez", "MAD SD ex", "MAD SD ez", "80% 타원", "예측 SD 좌우 cm", "예측 SD 높이 cm"]].round(3))

,2019,2020
공,13458.0000,12615.0000
24칸 ll,2.8933,2.8967
llx,0.0134,0.0213
포함률,0.8151,0.8103
기대 포함률,0.8162,0.8160
C 예측,0.4128,0.3948
C 실제,0.3925,0.3853
rC,-0.0203,-0.0094
rH,0.0099,0.0028
평균 ex,0.0090,0.0032


구종별 (2019)


,공,llx,rC,평균 ex,평균 ez,SD ex,SD ez,MAD SD ex,MAD SD ez,80% 타원,예측 SD 좌우 cm,예측 SD 높이 cm
CHUP,1654.0,0.029,-0.020,0.018,0.082,0.987,0.983,0.935,0.943,0.821,21.162,29.505
CURV,1190.0,0.003,-0.024,-0.016,0.060,0.977,0.986,0.991,0.961,0.818,25.625,33.799
CUTT,57.0,0.106,-0.069,0.175,-0.112,1.125,0.918,0.828,0.783,0.789,20.226,28.864
FAST,6627.0,0.002,-0.015,0.011,0.083,0.983,0.992,0.995,0.981,0.817,24.718,26.349
FORK,720.0,0.061,-0.060,0.003,0.099,0.964,0.959,0.952,0.900,0.817,21.836,31.640
SLID,2458.0,0.038,-0.025,0.014,0.103,1.008,0.981,0.988,0.937,0.809,22.734,30.879
TWOS,752.0,-0.037,-0.006,-0.011,0.052,0.942,0.988,0.928,0.950,0.826,23.153,24.814


#### 🔍 0-1-2 결과: 전체 보정·퍼짐은 거의 맞음
- 포함률 0.815 / 기대 0.816 (2020 0.810 / 0.816), llx +0.013 / +0.021 (아주 조금 과신)
- 퍼짐: SD ex·ez 0.98–1.01, 80% 타원 0.816 / 0.804 -> 평균 퍼짐 크기는 맞음, 구종별도 0.81–0.83 (공 57개인 커터 빼고)
  - MAD SD가 SD보다 조금 작음 (2019 높이 0.957 vs 0.986) -> 가운데는 조금 넓고 꼬리는 조금 두꺼움 (정규분포 모양 몫)
- C 과다 예측: rC 2019 −0.020, 2020 −0.009 (README 4-4 model_03 숫자와 같음)
- 높이 중심: 2019 평균 ez +0.083 (실제가 예측보다 약 2.4cm 높음), 2020 +0.011 -> 2019 리그 추세(4-3)와 같은 해 효과

#### 0-1-3. 투수별

- 투수별 평균과 경기 단위 표준오차 (같은 경기 공끼리 독립이 아님) -> z = 평균 / 표준오차 (80% 타원은 0.8과의 차이)
- 모델이 맞고 투수 차이가 잡음뿐이면 z² 합 ≈ 투수 수
- 지속성: 같은 투수가 2019·2020 같은 방향인지 (상관이 0 근처면 투수 고유 특성 아님 -> 투수별 보정 불필요)

In [4]:
def game_se(d, col, center=0.0):  # 투수별 (col − center) 평균, 경기 단위 표준오차
    v = d[col] - center
    gg = v.groupby([d["pitcher_code"], d["gameID"]]).agg(["sum", "count"])
    r = gg.groupby(level=0).agg(s=("sum", "sum"), n=("count", "sum"))
    mean = r.s / r.n
    resid = gg["sum"] - gg["count"] * mean.reindex(gg.index.get_level_values(0)).to_numpy()
    se = np.sqrt((resid ** 2).groupby(level=0).sum()) / r.n
    return mean, se, r.n

def per_pitcher(d):  # 투수별 표 (gap 낮은 순)
    out = {}
    for col, ctr in [("gap", 0.0), ("rC", 0.0), ("llx", 0.0), ("in80", 0.8)]:
        mean, se, n = game_se(d, col, ctr)
        out[col], out["z " + col] = mean, mean / se
    out = pd.DataFrame(out)
    out.insert(0, "초구", n)
    out.insert(0, "투수", names.reindex(out.index).to_numpy())
    out = out.rename(columns={"in80": "80% 타원 − 0.8", "z in80": "z 타원"})
    g = d.groupby("pitcher_code")
    out["평균 ez"], out["SD ex"], out["SD ez"] = g.ez.mean(), g.ex.std(), g.ez.std()
    return out.sort_values("gap")

PP = {yr: per_pitcher(D[yr]) for yr in D}
for yr, t in PP.items():
    zs = {c: (t[c] ** 2).sum() for c in ["z gap", "z rC", "z llx", "z 타원"]}
    print(f"{yr}: 투수 {len(t)}명, z² 합 " + " / ".join(f"{c[2:]} {v:.1f}" for c, v in zs.items()))
    print(f"   |z gap| > 2: {', '.join(t.loc[t['z gap'].abs() > 2, '투수'])}")
    print(f"   |z rC| > 2: {', '.join(t.loc[t['z rC'].abs() > 2, '투수'])}")
print("\n2019 (gap 낮은 순)")
display(PP[fc.VAL_YEAR].round(3))

cols = ["gap", "rC", "llx", "80% 타원 − 0.8", "평균 ez", "SD ex", "SD ez"]
j = PP[fc.VAL_YEAR][["투수"] + cols].join(PP[fc.TEST_YEAR][cols], lsuffix=" 2019", rsuffix=" 2020", how="inner")
print(f"\n지속성 (2019 vs 2020 투수별 상관, 투수 {len(j)}명)")
display(pd.Series({c: j[c + " 2019"].corr(j[c + " 2020"]) for c in cols}, name="상관").round(2).to_frame().T)

2019: 투수 38명, z² 합 gap 59.2 / rC 77.8 / llx 67.1 / 타원 95.6
   |z gap| > 2: 장현식, 주권, 최원준
   |z rC| > 2: 안우진, 이태양, 양현종, 알칸타라
2020: 투수 39명, z² 합 gap 90.2 / rC 70.2 / llx 74.1 / 타원 83.5
   |z gap| > 2: 원태인, 이민우, 김재윤, 이태양, 최원준, 안우진, 김진성
   |z rC| > 2: 함덕주, 임찬규, 이용찬, 박상원, 이재학, 안우진

2019 (gap 낮은 순)


,투수,초구,gap,z gap,rC,z rC,llx,z llx,80% 타원 − 0.8,z 타원,평균 ez,SD ex,SD ez
pitcher_code,,,,,,,,,,,,,
63950,장현식,218,-0.077,-3.438,0.028,0.869,0.110,2.694,-0.043,-1.539,0.268,1.026,1.027
66920,최성영,365,-0.048,-1.949,-0.032,-1.708,0.099,2.668,0.014,0.749,0.126,0.957,1.047
67703,박상원,218,-0.037,-1.282,-0.050,-1.585,0.073,1.273,0.039,1.553,0.070,0.987,0.904
63512,박진형,132,-0.034,-0.736,-0.056,-1.334,0.065,0.921,0.033,1.014,0.267,0.962,0.908
61101,임찬규,347,-0.033,-1.652,0.001,0.047,0.113,2.675,-0.039,-1.641,0.055,1.065,1.071
63248,함덕주,189,-0.032,-1.117,0.016,0.466,0.020,0.317,-0.012,-0.398,-0.002,1.045,1.097
69446,원태인,467,-0.027,-1.353,-0.006,-0.283,0.029,0.716,0.022,1.307,0.073,0.960,1.004
65048,김민수,345,-0.024,-0.976,0.002,0.095,0.038,1.082,-0.014,-0.502,-0.001,0.994,1.055
67828,김성민,210,-0.022,-0.696,-0.012,-0.373,0.088,1.550,-0.014,-0.555,0.081,1.076,0.990



지속성 (2019 vs 2020 투수별 상관, 투수 37명)


,gap,rC,llx,80% 타원 − 0.8,평균 ez,SD ex,SD ez
상관,0.18,-0.28,0.1,-0.14,-0.14,0.13,0.12


#### 🔍 0-1-3 결과: 투수 고유 문제 아님
- 해 안에서는 투수 차이가 잡음보다 큼 (2019 z² 합 gap 59 · rC 78 · llx 67 · 타원 96 / 투수 38명)
- 그런데 해마다 이어지지 않음: 2019 vs 2020 투수별 상관 gap 0.18 · rC −0.28 · llx 0.10 · 타원 −0.14 · SD ex 0.13 · SD ez 0.12
  - rC |z| > 2: 2019 안우진·이태양·양현종·알칸타라 / 2020 함덕주·임찬규·이용찬·박상원·이재학·안우진 -> 안우진만 겹침
  - 투수별 퍼짐(80% 타원)도 그 해 안에서만 차이 (2019 박세웅 +0.066 · 알칸타라 +0.054 넓음, 장현식 −0.043 좁음)
- -> 투수별 보정 안 함 (구종 4-6과 같은 결론)

#### 0-1-4. 기록 수·시즌 초·역할별

- 1단계 보완 후보와 연결: 기록 수 구간 = 당김 강도 / 시즌 몇 번째 등판·월 = 시즌 초 / SD = 분산 따로 당기기
- 오른쪽 열은 2020 (같은 방향인지)

In [5]:
def cut(s, b, lab=None):
    return pd.cut(s, b, labels=lab, right=False)

AGG = {"n": ("ll", "size"), "24칸 ll": ("ll", "mean"), "llx": ("llx", "mean"), "gap": ("gap", "mean"), "rC": ("rC", "mean"),
       "평균 ex": ("ex", "mean"), "평균 ez": ("ez", "mean"), "SD ex": ("ex", "std"), "SD ez": ("ez", "std"), "80% 타원": ("in80", "mean")}

def table(d, cols, agg=AGG):  # 그룹 변수마다 칸별 요약을 세로로 이어 붙임
    rows = []
    for var, c in cols.items():
        key = pd.Series(np.asarray(c), index=d.index, name="값")
        t = d.groupby(key, observed=True).agg(**agg)
        t.index = pd.MultiIndex.from_product([[var], t.index.astype(str)], names=["변수", "값"])
        rows.append(t)
    return pd.concat(rows)

def rec_cols(d):  # 기록 수·시즌 초·역할 그룹
    return {"투수×구종×손 전날까지 초구 수": cut(d["n_hand"], [0, 1, 31, 101, 301, 1e9], ["0", "1–30", "31–100", "101–300", "301+"]),
            "그 시즌 몇 번째 등판": cut(d["app_no"], [1, 2, 4, 11, 999], ["1", "2–3", "4–10", "11+"]),
            "월 (3월은 4월, 10월은 9월에)": d["game_date"].dt.month.clip(4, 9),
            "선발/불펜": np.where(d["starter"].eq(1), "선발", "불펜"),
            "타자 손": d["same_side"].map({0: "반대 손", 1: "같은 손"})}

T19, T20 = table(D[fc.VAL_YEAR], rec_cols(D[fc.VAL_YEAR])), table(D[fc.TEST_YEAR], rec_cols(D[fc.TEST_YEAR]))
display(T19.join(T20[["n", "gap", "rC", "평균 ez", "SD ex", "SD ez", "80% 타원"]], rsuffix=" 2020").round(3))

n  24칸 ll    llx    gap     rC  평균 ex  평균 ez  SD ex  SD ez  80% 타원  n 2020  gap 2020  rC 2020  평균 ez 2020  SD ex 2020  SD ez 2020  80% 타원 2020
변수                   값                                                                                                                                                         
투수×구종×손 전날까지 초구 수    0          89   2.886  0.009 -0.006 -0.044  0.228 -0.030  0.926  1.004   0.854    34.0    -0.139    0.119       0.340       1.291       1.211        0.706
                     101–300  5672   2.893  0.005  0.003 -0.011  0.009  0.095  0.991  0.990   0.816  5826.0    -0.001   -0.011      -0.009       0.988       0.990        0.812
                     1–30     1620   2.882  0.033 -0.016 -0.024  0.017  0.034  0.980  1.008   0.807   729.0    -0.012   -0.001       0.008       1.204       1.126        0.770
                     301+     2461   2.940  0.010  0.000 -0.029 -0.040  0.089  0.964  0.992   0.813  3671.0    -0.010   -0.008       0.014       0.992       1.031        0.802
                     31–100   3616   2.868  0.021 -0.001 -0.026  0.033  0.084  0.991  0.965   0.822  2355.0    -0.005   -0.012       0.053       1.014       1.002        0.799
그 시즌 몇 번째 등판         1         419   2.956  0.078 -0.030 -0.071  0.037  0.043  0.944  0.975   0.833   455.0    -0.020   -0.036      -0.083       1.011       1.001        0.813
                     11+      8741   2.885  0.005  0.003 -0.017  0.009  0.082  0.986  0.985   0.816  8106.0    -0.003   -0.014       0.028       0.991       0.996        0.810
                     2–3       925   2.911  0.031 -0.019 -0.012  0.032  0.029  1.012  0.953   0.819   934.0    -0.032   -0.008      -0.046       1.027       0.991        0.802
                     4–10     3373   2.903  0.022 -0.002 -0.026 -0.002  0.105  0.980  0.999   0.815  3120.0    -0.003    0.005      -0.001       1.047       1.063        0.788
월 (3월은 4월, 10월은 9월에) 4        2568   2.925  0.045 -0.012 -0.026  0.025  0.104  0.999  0.987   0.811     NaN       NaN      NaN         NaN         NaN         NaN          NaN
                     5        2403   2.906  0.022  0.001 -0.025  0.014  0.062  0.982  1.003   0.807  2264.0    -0.023   -0.013      -0.035       1.017       0.999        0.798
                     6        2493   2.903  0.025 -0.001 -0.026  0.054  0.088  0.986  0.988   0.821  2379.0    -0.004    0.001       0.048       1.020       1.026        0.787
                     7        1791   2.851 -0.028  0.005  0.008 -0.001  0.076  0.971  0.993   0.822  2233.0    -0.004   -0.012       0.017       1.032       1.063        0.809
                     8        2362   2.897  0.015 -0.005 -0.022 -0.010  0.118  1.011  0.978   0.811  1985.0    -0.002   -0.014       0.053       0.978       1.002        0.813
                     9        1841   2.856 -0.019  0.010 -0.024 -0.047  0.035  0.944  0.963   0.831  3754.0     0.001   -0.009      -0.010       0.997       0.988        0.811
선발/불펜                불펜       4242   2.886  0.019 -0.000 -0.021  0.015  0.097  1.002  0.987   0.813  3979.0     0.004   -0.007       0.008       1.001       0.993        0.807
                     선발       9216   2.897  0.011 -0.002 -0.020  0.006  0.076  0.977  0.986   0.818  8636.0    -0.010   -0.010       0.012       1.012       1.023        0.803
타자 손                 같은 손     6941   2.908  0.010  0.001 -0.019  0.018  0.092  0.981  0.989   0.817  6372.0    -0.004   -0.011      -0.007       1.020       1.019        0.807
                     반대 손     6517   2.878  0.017 -0.004 -0.021 -0.001  0.073  0.989  0.983   0.815  6243.0    -0.007   -0.008       0.030       0.997       1.007        0.800

#### 🔍 0-1-4 결과: 시즌 첫 등판들에서 어긋남, 기록 수·역할은 대체로 맞음
- 그 시즌 몇 번째 등판 (2019 / 2020)
  - 1번째: gap −0.030 / −0.020, rC −0.071 / −0.036, llx +0.078 (2019, 과신), 2019 24칸 ll 2.956 (11번째 이후 2.885)
  - 2–3번째: gap −0.019 / −0.032
  - 4번째부터 거의 0 -> 두 해 같은 방향 -> **시즌 초 배수 근거** (첫 3등판 공은 약 10%라 전체 크기는 작을 것)
- 기록 수: 31구 이상은 SD·80% 타원 모두 맞음
  - 1–30구: 2019 과신 (llx +0.033), 2020 퍼짐이 좁음 (SD 1.20 · 1.13, 80% 타원 0.77) / 0구는 공 89 · 34개라 판단 어려움
  - 1단계 진단(위치 당김 강도는 기록 수별로 ×1이 맞음)과 대체로 같음, 기록 적은 쪽 퍼짐은 분산 따로 당기기에서 같이 확인
- 월: 4월 llx +0.045로 조금 과신, 그 밖은 방향 없음 / 선발·불펜, 타자 손: 차이 없음

#### 0-1-5. 상황별 (위치 모델은 상황을 모름) + 투수별 차이를 상황이 얼마나 설명하나

- 상황 표: 3단계 변수(5장 3번 변수 4·6) 근거, 오른쪽 열은 2020
- 분해: 상황 칸 = 첫 타자 여부 × 재대결 여부 × 아웃 × 주자(없음·1루만·득점권) (36칸)
  - 상황몫 = 그 투수 공들의 상황 칸 평균 rC (모든 투수로 계산) -> 상황 구성만으로 기대되는 rC, 남은몫 = rC − 상황몫
  - 같은 해 안에서 계산한 진단 (모델 아님)

In [6]:
def runner(d):  # 주자 상황 3칸
    return np.select([(d["base2"] + d["base3"]) > 0, d["base1"] > 0], ["득점권", "1루만"], "없음")

def sit_cols(d):
    return {"앞선 투구 수": cut(d["pc"], [0, 1, 16, 31, 61, 999], ["0 (첫 타자)", "1–15", "16–30", "31–60", "61+"]),
            "같은 타자 몇 번째": d["tto"].clip(upper=3).map({1: "1", 2: "2", 3: "3+"}),
            "아웃": d["out_cn"].clip(upper=2),
            "주자": runner(d),
            "점수 차 (투수 팀)": cut(d["sdiff"], [-99, -3, 0, 1, 4, 99], ["−3↓", "−2~−1", "0", "+1~+3", "+4↑"]),
            "이닝": d["inning"].clip(upper=9),
            "타순": d["batorder"]}

AGG_S = {"n": ("ll", "size"), "llx": ("llx", "mean"), "gap": ("gap", "mean"), "rC": ("rC", "mean"), "rH": ("rH", "mean"),
         "평균 ex": ("ex", "mean"), "평균 ez": ("ez", "mean")}
S19, S20 = table(D[fc.VAL_YEAR], sit_cols(D[fc.VAL_YEAR]), AGG_S), table(D[fc.TEST_YEAR], sit_cols(D[fc.TEST_YEAR]), AGG_S)
S19["rC se"] = np.sqrt(0.25 / S19["n"])  # 공 단위 근사 (C 비율 약 0.4 -> 분산 ≤ 0.25)
display(S19.join(S20[["n", "rC", "평균 ez"]], rsuffix=" 2020").round(3))

def decompose(d):  # 투수별 rC = 상황몫 + 남은몫
    d = d.copy()
    d["cell"] = (d["pc"].eq(0).astype(str) + "|" + (d["tto"] >= 2).astype(str) + "|" + d["out_cn"].clip(upper=2).astype(str) + "|" + runner(d))
    d["rC_ctx"] = d.groupby("cell")["rC"].transform("mean")
    p = d.groupby("pitcher_code").agg(n=("rC", "size"), rC=("rC", "mean"), 상황몫=("rC_ctx", "mean"),
                                      선발비율=("starter", "mean"))
    p["득점권비율"] = d.assign(sp=runner(d) == "득점권").groupby("pitcher_code")["sp"].mean()  # 득점권 초구 비율
    p["남은몫"] = p["rC"] - p["상황몫"]
    p.insert(0, "투수", names.reindex(p.index).to_numpy())
    return p.sort_values("rC")

DEC = {yr: decompose(D[yr]) for yr in D}
for yr, p in DEC.items():
    print(f"{yr}: 투수별 rC 분산 {p.rC.var():.5f}, 상황몫 {p['상황몫'].var():.5f}, 남은몫 {p['남은몫'].var():.5f}, corr(rC, 상황몫) {p.rC.corr(p['상황몫']):.2f}")
jj = DEC[fc.VAL_YEAR][["rC", "상황몫", "남은몫"]].join(DEC[fc.TEST_YEAR][["rC", "상황몫", "남은몫"]], lsuffix=" 2019", rsuffix=" 2020", how="inner")
print("지속성 (2019 vs 2020 투수별 상관): " + ", ".join(f"{c} {jj[c + ' 2019'].corr(jj[c + ' 2020']):.2f}" for c in ["rC", "상황몫", "남은몫"]))
print("\n2019 rC 낮은 5명 / 높은 5명")
p = DEC[fc.VAL_YEAR]
display(pd.concat([p.head(5), p.tail(5)]).round(3))

n    llx    gap     rC     rH  평균 ex  평균 ez  rC se  n 2020  rC 2020  평균 ez 2020
변수          값                                                                                           
앞선 투구 수     0 (첫 타자)  1288  0.039 -0.009 -0.033  0.010 -0.057  0.152  0.014    1278   -0.005       0.080
            16–30     2243  0.013  0.001 -0.037  0.010 -0.005  0.120  0.011    2012    0.005       0.019
            1–15      3868  0.007  0.004 -0.011  0.009  0.020  0.013  0.008    3718   -0.002      -0.038
            31–60     3220  0.007 -0.002 -0.019  0.011  0.018  0.103  0.009    2911   -0.029       0.032
            61+       2839  0.019 -0.005 -0.016  0.009  0.024  0.094  0.009    2696   -0.011       0.017
같은 타자 몇 번째  1         7645  0.013  0.002 -0.022  0.010 -0.004  0.069  0.006    7390   -0.002       0.002
            2         3534  0.014 -0.005 -0.022  0.014  0.021  0.105  0.008    3276   -0.032       0.035
            3+        2279  0.015 -0.007 -0.010  0.004  0.034  0.095  0.010    1949   -0.001       0.006
아웃          0         4645  0.042 -0.011 -0.033  0.011 -0.005  0.175  0.007    4375   -0.024       0.077
            1         4500 -0.002  0.005 -0.013  0.008  0.028  0.028  0.007    4222   -0.017      -0.009
            2         4313 -0.001  0.003 -0.014  0.011  0.004  0.041  0.008    4018    0.015      -0.039
주자          1루만       2886  0.024 -0.003 -0.026  0.019  0.028  0.080  0.009    2723    0.002      -0.003
            득점권       3346 -0.022  0.006  0.032 -0.027  0.049  0.012  0.009    3136    0.024      -0.077
            없음        7226  0.025 -0.003 -0.042  0.023 -0.017  0.116  0.006    6756   -0.029       0.058
점수 차 (투수 팀) +1~+3     3814  0.008  0.001 -0.018  0.013 -0.000  0.131  0.008    3558   -0.003       0.029
            +4↑       1428  0.034 -0.008 -0.035  0.009  0.001  0.172  0.013    1904   -0.031       0.065
            0         3715  0.005  0.003 -0.021  0.009 -0.011  0.045  0.008    3213   -0.015      -0.004
            −2~−1     3521  0.011 -0.002 -0.019  0.008  0.028  0.044  0.008    3109    0.005      -0.021
            −3↓        980  0.042 -0.009 -0.010  0.011  0.064  0.046  0.016     831   -0.018      -0.009
이닝          1         1768 -0.003  0.011 -0.015  0.008  0.016 -0.071  0.012    1626    0.007      -0.090
            2         1680  0.032 -0.014 -0.029  0.012 -0.024  0.107  0.012    1596    0.007       0.059
            3         1710  0.005 -0.003 -0.023  0.009  0.016  0.093  0.012    1618   -0.010       0.036
            4         1666 -0.011  0.018 -0.016  0.028  0.042  0.092  0.012    1621   -0.019       0.006
            5         1616  0.026 -0.015 -0.008 -0.001  0.012  0.094  0.012    1535   -0.024       0.029
            6         1517 -0.008  0.011 -0.009 -0.002 -0.006  0.134  0.013    1308   -0.008       0.047
            7         1289  0.033 -0.002 -0.039  0.018 -0.002  0.126  0.014    1133   -0.024      -0.007
            8         1226  0.066 -0.027 -0.034  0.003  0.047  0.101  0.014     990    0.011      -0.030
            9          986 -0.012  0.008 -0.012  0.014 -0.029  0.109  0.016    1188   -0.025       0.048
타순          1         1643  0.032 -0.000 -0.038  0.041 -0.003  0.086  0.012    1550   -0.042       0.058
            2         1597  0.013  0.010 -0.033  0.013  0.002  0.097  0.013    1505   -0.008      -0.058
            3         1550 -0.004  0.001 -0.015  0.020  0.048  0.050  0.013    1500    0.015      -0.060
            4         1526  0.024  0.001 -0.012 -0.010  0.108  0.016  0.013    1472    0.009       0.002
            5         1513  0.010 -0.011  0.001  0.004  0.020  0.020  0.013    1399    0.004      -0.003
            6         1480  0.009 -0.006  0.001 -0.010  0.005  0.065  0.013    1364   -0.026       0.031
            7         1439 -0.011 -0.002  0.003 -0.004 -0.021  0.110  0.013    1320   -0.014       0.079
            8         1396  0.032 -0.000 -0.038  0.008 -0.028  0.155  0.013    1282   -0.007       0.036
            9         1314  0.013 -

2019: 투수별 rC 분산 0.00110, 상황몫 0.00002, 남은몫 0.00110, corr(rC, 상황몫) 0.04
2020: 투수별 rC 분산 0.00161, 상황몫 0.00006, 남은몫 0.00156, corr(rC, 상황몫) 0.17
지속성 (2019 vs 2020 투수별 상관): rC -0.28, 상황몫 -0.05, 남은몫 -0.30

2019 rC 낮은 5명 / 높은 5명


,투수,n,rC,상황몫,선발비율,득점권비율,남은몫
pitcher_code,,,,,,,
60768,이태양,231,-0.095,-0.016,0.100,0.316,-0.079
68341,안우진,371,-0.086,-0.018,0.960,0.275,-0.068
65062,김재윤,160,-0.062,-0.032,0.000,0.150,-0.030
63512,박진형,132,-0.056,-0.024,0.000,0.242,-0.032
65616,이민우,269,-0.056,-0.011,0.491,0.372,-0.044
63248,함덕주,189,0.016,-0.018,0.000,0.312,0.034
61643,홍건희,381,0.016,-0.018,0.832,0.241,0.034
63950,장현식,218,0.028,-0.013,0.000,0.376,0.041
67119,고우석,221,0.063,-0.029,0.000,0.208,0.091


#### 🔍 0-1-5 결과: 상황 신호는 있지만 투수별 차이는 설명 못 함
- 주자: rC 없음 −0.042 / 득점권 +0.032 (2020 −0.029 / +0.024) -> 4-9과 같음, 득점권에서 존 밖으로 더 뺌
  - 높이: 주자 없음 평균 ez +0.116 vs 득점권 +0.012 (2020 +0.058 vs −0.077) -> 주자 없을 때 더 높게
- 그 밖 두 해 같은 방향: 0아웃 rC −0.033 / −0.024, 점수 차 +4 이상 −0.035 / −0.031, 첫 타자 −0.033 / −0.005 (2020 약함)
  - 타순별 좌우: 2019 4번 평균 ex +0.108, 9번 −0.065 (2020은 이 표에 없음)
- -> 3단계 변수(5장 3번 변수 4·6) 근거, 1단계 보완 대상은 아님
- 투수별 rC는 거의 전부 남은몫 (2019 상황몫 분산 0.00002 vs rC 0.0011, 상관 0.04 / 2020 0.17) -> 투수마다 상황 구성이 달라서 생긴 차이 아님, 남은몫도 해마다 안 이어짐 (상관 −0.30)

#### 0-1-6. 실전: 구종을 몰라서 잃는 몫 (1단계 메모 "기대 0.058 vs 실제 0.048")

- 실제로 잃은 몫 = 실전 24칸 log loss − 실제 구종 기준 / 모델이 기대한 몫 = Σ P(구종)·KL(P(칸|구종) || 섞은 분포) (model_01 7장과 같음)
- 기대 > 실제 -> 모델이 구종별 위치 분포 차이를 실제보다 크게 잡음 (구종마다 분포가 너무 날카롭거나 서로 너무 다름)
- 구종 확률은 model_09 (1단계 메모 숫자는 model_01 구종 확률이라 조금 다를 수 있음)

In [7]:
def lost(d):
    return pd.Series({"공": len(d), "실전 ll": d.ll_mix.mean(), "실제 구종 ll": d.ll.mean(), "실제로 잃은 몫": (d.ll_mix - d.ll).mean(),
                      "모델이 기대한 몫": d.kl.mean(), "C 예측 (실전)": d.pC_mix.mean(), "C 실제": (d.rC + d.pC).mean()})

display(pd.DataFrame({yr: lost(D[yr]) for yr in D}).round(4))
for yr in D:
    d = D[yr]
    t = pd.DataFrame({k: lost(g) for k, g in d.groupby(pd.cut(d.p_true_type, [0, .05, .2, .5, 1.0]), observed=True)}).T
    t2 = pd.DataFrame({k: lost(g) for k, g in d.groupby(cut(d.n_hand, [0, 1, 31, 101, 301, 1e9], ["0", "1–30", "31–100", "101–300", "301+"]), observed=True)}).T
    print(yr, "· 실제 구종에 준 확률 / 투수×구종×손 기록 수")
    display(pd.concat({"실제 구종 확률": t, "기록 수": t2}).round(4))

,2019,2020
공,13458.0000,12615.0000
실전 ll,2.9493,2.9454
실제 구종 ll,2.8933,2.8967
실제로 잃은 몫,0.0561,0.0488
모델이 기대한 몫,0.0608,0.0611
C 예측 (실전),0.4126,0.3952
C 실제,0.3925,0.3853


2019 · 실제 구종에 준 확률 / 투수×구종×손 기록 수


공   실전 ll  실제 구종 ll  실제로 잃은 몫  모델이 기대한 몫  C 예측 (실전)    C 실제
실제 구종 확률 (0.0, 0.05]   347.0  2.9636    2.8444    0.1192     0.0586     0.4107  0.4063
         (0.05, 0.2]  1944.0  2.9453    2.8733    0.0719     0.0642     0.4138  0.4203
         (0.2, 0.5]   6407.0  2.9475    2.8806    0.0669     0.0669     0.4141  0.4005
         (0.5, 1.0]   4760.0  2.9525    2.9220    0.0305     0.0513     0.4102  0.3693
기록 수     0              89.0  2.9836    2.8857    0.0979     0.0526     0.4117  0.3708
         1–30         1620.0  2.9438    2.8820    0.0618     0.0621     0.4140  0.4123
         31–100       3616.0  2.9270    2.8677    0.0593     0.0662     0.4116  0.4043
         101–300      5672.0  2.9489    2.8927    0.0561     0.0570     0.4179  0.3977
         301+         2461.0  2.9857    2.9397    0.0460     0.0610     0.4009  0.3507

2020 · 실제 구종에 준 확률 / 투수×구종×손 기록 수


공   실전 ll  실제 구종 ll  실제로 잃은 몫  모델이 기대한 몫  C 예측 (실전)    C 실제
실제 구종 확률 (0.0, 0.05]   325.0  2.9672    3.0760   -0.1087     0.0627     0.3958  0.4492
         (0.05, 0.2]  1751.0  2.9473    2.8598    0.0875     0.0644     0.3951  0.4432
         (0.2, 0.5]   6035.0  2.9473    2.8790    0.0683     0.0680     0.3994  0.3876
         (0.5, 1.0]   4504.0  2.9406    2.9217    0.0189     0.0505     0.3895  0.3552
기록 수     0              34.0  3.0114    2.9675    0.0439     0.0605     0.3995  0.5294
         1–30          729.0  2.9378    2.9631   -0.0254     0.0610     0.3973  0.4307
         31–100       2355.0  2.9333    2.8585    0.0747     0.0720     0.4017  0.4098
         101–300      5826.0  2.9347    2.8834    0.0513     0.0604     0.3958  0.3828
         301+         3671.0  2.9712    2.9283    0.0429     0.0552     0.3895  0.3634

#### 🔍 0-1-6 결과: 모델이 기대한 몫이 실제보다 큼, 기록 많은 투수에서
- 실제로 잃은 몫 vs 모델이 기대한 몫: 2019 0.056 vs 0.061, 2020 0.049 vs 0.061 (1단계 메모 0.048 vs 0.058과 같은 모습)
- 기록 수별 (그 공 앞 정보라 나눠 봐도 됨, 2019 / 2020)
  - 301구 이상: 실제 0.046 / 0.043 vs 기대 0.061 / 0.055
  - 101–300구: 실제 0.056 / 0.051 vs 기대 0.057 / 0.060
  - -> 기록 많은 투수일수록 구종별 위치 분포 차이를 실제보다 크게 잡음 / 원인 추정 (확인 안 함): 반감 35.4구로 최근 공만 보니 드문 구종 분포가 표본 잡음으로 서로 달라 보임
- 실제 구종 확률별 표는 결과(실제 구종)로 나눈 것이라 기대 몫과 직접 비교하지 않음
- C (실전): 예측 0.413 / 0.395 vs 실제 0.393 / 0.385, 2019 301구 이상에서 가장 큼 (0.401 vs 0.351)
- 0-2에서 분산 따로 당기기·모든 카운트(더 많은 공으로 구종별 분포 안정)가 이 차이를 줄이는지 같이 봄

#### 🔍 0-1 결론 (1단계 보완 후보별)
- **분산 따로 당기기**: 평균 퍼짐은 이미 맞음 (80% 타원 0.816 / 0.804) -> 큰 이득은 기대 어려움, 기록 1–30구와 기대 몫 차이(0-1-6)에서만 여지
- **시즌 초 배수**: 시즌 첫 1–3등판에서 두 해 같은 방향으로 어긋남 -> 근거 있음 (크기는 작을 것)
- **모든 카운트 위치 성향**: 진단으로는 직접 못 봄, 기록 많은 투수의 기대 몫 차이가 단서 -> 0-2에서 잼
- **투수별 적중**: 투수 고유 아님 -> 따로 고치지 않음
- 상황 신호(주자·아웃·점수 차·높이)는 3단계 변수 단계로

## 모델

In [8]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 구종은 model_09 값 고정 (구종 B4, 변수 `FEAT`, LightGBM 잎 7·잎당 최소 20·λ 1000·트리 730·월 단위 재학습)
- 위치: 출발 값은 model_09 (= model_03) + 시즌 초 배수 1 (= 없음) -> 3-1에서 다시 고른 값으로 바꿈

In [9]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치: 출발 값 = model_09 (model_03) -> 3-1에서 다시 고름
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구 (고정)
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 17.3  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k
LOC_SEASON_F = 1.0  # 시즌 초 배수: 새 시즌 첫 경기 전에 지난 시즌까지 투수 위치 누적 무게에 곱함 (1 = model_09)
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [10]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [11]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [12]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_11    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_11    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: 시즌 초 배수

- `fc.prior_table`·`fc.lookup`의 `season_f` (model_06 구종 B4와 같은 함수): 새 시즌 첫 경기 전에 지난 시즌까지 누적 S·N에 곱함
  - 투수×구종·투수×구종×손 두 표에 같은 배수 / 기준점(분석 대상 투수 구종×손, 반감 173구 = 며칠치)은 시즌 경계 영향이 작아 그대로
  - 배수가 작을수록 시즌 첫 경기들에서 지난 시즌 기록을 덜 믿고 기준점 쪽으로 감, 이번 시즌 공이 쌓이면 원래대로
- 확인: `season_f = 1`이면 model_09과 같음 (재현)

In [14]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0):  # 위치 모멘트 + 24칸 비율 (n, 29): 기준점 -> 투수×구종 (k_type) -> 투수×구종×손 (k_hand)
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본 (t: 이름 하나 또는 공마다 배열)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f), qt)  # 모멘트 -> 정규분포 (높이는 그 타자 존으로 cm 환산)
    return zone_probs(g, qt, cov), g  # 정규분포를 24칸 위에서 적분 (공분산 반영)

# 재현: season_f = 1 -> model_09 실제 구종 기준 24칸 log loss
yz_v = q_val["zone"].map(fc.ZONE_IDX).to_numpy()
P0, _ = loc_given_type(q_val, fc.VAL_YEAR, q_val["pitch_type"].to_numpy(), LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, 1.0)
ll0 = -np.log(np.clip(P0[np.arange(len(q_val)), yz_v], 1e-12, None))
assert np.allclose(ll0, fc.align(m09, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy()), "season_f = 1인데 model_09과 다름"
print(f"season_f = 1 -> 2019 실제 구종 기준 24칸 log loss {ll0.mean():.5f} = model_09 (재현 확인)")

season_f = 1 -> 2019 실제 구종 기준 24칸 log loss 2.89326 = model_09 (재현 확인)


### 3-1. 시즌 초 배수 + 위치 파라미터 다시 고르기 (2019)

- 연결된 파라미터를 함께: 시즌 초 배수 × 반감기 × k_type × k_hand (기준점은 고정)
  - 구종 model_06에서 시즌 초 배수를 넣자 더 긴 반감기가 좋아짐 (지난 시즌 꼬리를 끊으니 시즌 안 기록을 더 길게 봄) -> 반감기도 같이
- 거친 격자 전부 -> 1등이 범위 끝이면 넓혀 다시 (시즌 초 배수는 1이 위쪽 끝) -> 1등 주변 세밀한 격자 전부
- 대조군 = model_09 값 (시즌 초 배수 1, 35.4 · 1000 · 17.3), 거친 격자에 들어 있음
- 기준: 2019 실제 구종 기준 24칸 log loss (README 3장), 1등 vs 대조군 경기 단위 부트스트랩

In [15]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN -> None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

def game_ci(d, gid, B=1000):  # 경기 단위 부트스트랩 95% 구간 (같은 경기 공은 독립이 아님)
    s = pd.DataFrame({"g": gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

REF19 = loc_league(q_val, fc.VAL_YEAR, LOC_REF, LOC_REF_HL)  # 2019 기준점 (고정)
_LSN = {}  # (반감기, 시즌 초 배수) -> 2019 S·N (prior_table은 캐시 안 함: 조합이 많아 메모리)

def ll_sf(season_f, half_life, k_type, k_hand):  # 2019 공별 24칸 log loss (실제 구종)와 정규분포
    key = (half_life, season_f)
    if key not in _LSN:
        _LSN[key] = loc_sn(q_val, half_life, season_f, cache=False)
    (S1, N1), (S2, N2) = _LSN[key]
    g = fc.gauss(fc.shrink(S2, N2, fc.shrink(S1, N1, REF19, k_type), k_hand), q_val)
    P = zone_probs(g, q_val)
    return -np.log(np.clip(P[np.arange(len(q_val)), yz_v], 1e-12, None)), g

def score_sf(season_f, half_life, k_type, k_hand):  # 한 조합의 2019 채점
    ll, g = ll_sf(season_f, half_life, k_type, k_hand)
    return {"logloss": ll.mean(), "좌표 NLL": fc.coord_metrics(g, q_val)["NLL"]}

CTL = dict(season_f=1.0, half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND)  # 대조군 = model_09
GRID_SF = {"season_f": [1.0, 0.5, 0.25, 0.1], "half_life": [25.0, 35.4, 50.0, 70.7],
           "k_type": [300.0, 1000.0, 3000.0], "k_hand": [10.0, 17.3, 30.0]}
res = grid_search(score_sf, GRID_SF)  # 거친 격자
tried = {k: list(v) for k, v in GRID_SF.items()}
for rnd in range(3):  # 1등이 범위 끝이면 바깥으로 넓힘 (시즌 초 배수 1은 위쪽 끝이라 넓히지 않음)
    b = best_of(res, GRID_SF)
    add = {}
    for k, v in b.items():
        if v == min(tried[k]):
            add[k] = [v / 2]
        elif v == max(tried[k]) and not (k == "season_f" and v >= 1.0):
            add[k] = [v * 2]
    if not add:
        break
    print(f"{rnd + 1}번째 넓히기:", add)
    for k, v in add.items():
        tried[k] += v
    res = pd.concat([res, grid_search(score_sf, {k: (add[k] if k in add else [b[k]]) for k in GRID_SF})])
    res = res.drop_duplicates(list(GRID_SF)).sort_values("logloss").reset_index(drop=True)
b = best_of(res, GRID_SF)
grid_f = {k: [v for v in finer(tried[k], b[k]) if not (k == "season_f" and v > 1.0)] for k in GRID_SF}  # 1등 주변 세밀한 후보 (배수는 1 이하)
print("거친 1등", b, "-> 세밀한 후보", grid_f)
res = pd.concat([res, grid_search(score_sf, grid_f)]).drop_duplicates(list(GRID_SF)).sort_values("logloss").reset_index(drop=True)

ctl = res[(res.season_f == 1.0) & (res.half_life == CTL["half_life"]) & (res.k_type == CTL["k_type"]) & (res.k_hand == CTL["k_hand"])].iloc[0]
s1 = res[res.season_f == 1.0].iloc[0]  # 시즌 초 배수 없이 다시 고른 1등 (파라미터만 다시 고른 효과)
print(f"2019 24칸 log loss (실제 구종): 대조군 model_09 {ctl.logloss:.5f} / 배수 1에서 1등 {s1.logloss:.5f} / 전체 1등 {res.logloss.iloc[0]:.5f}")
display(res.head(10).round(5))
for k, s in marginal(res, {k: sorted(set(tried[k]) | set(grid_f[k])) for k in GRID_SF}).items():
    print(k, s.round(5).to_dict())  # 파라미터별 값마다 1등 (민감도)

# 1등 vs 대조군, 배수 1 1등 vs 대조군: 경기 단위 부트스트랩 (양수 = 좋아짐)
gid_v = q_val["gameID"].to_numpy()
ll_ctl = ll_sf(**CTL)[0]
rows = {}
for nm, r in [("전체 1등", res.iloc[0]), ("배수 1에서 1등", s1)]:
    d = ll_ctl - ll_sf(**{k: float(r[k]) for k in GRID_SF})[0]
    lo, hi = game_ci(d, gid_v)
    rows[nm] = {**{k: float(r[k]) for k in GRID_SF}, "이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000}
display(pd.DataFrame(rows).T.round(4))

거친 1등 {'season_f': 0.5, 'half_life': 50.0, 'k_type': 1000.0, 'k_hand': 17.3} -> 세밀한 후보 {'season_f': [0.25, 0.354, 0.5, 0.707, 1.0], 'half_life': [35.4, 42.1, 50.0, 59.5, 70.7], 'k_type': [300.0, 548.0, 1000.0, 1730.0, 3000.0], 'k_hand': [10.0, 13.2, 17.3, 22.8, 30.0]}
2019 24칸 log loss (실제 구종): 대조군 model_09 2.89326 / 배수 1에서 1등 2.89326 / 전체 1등 2.89276


,season_f,half_life,k_type,k_hand,logloss,좌표 NLL
0,0.707,42.1,1000.0,17.3,2.89276,9.25182
1,0.707,42.1,548.0,17.3,2.89276,9.25192
2,0.707,42.1,1730.0,17.3,2.89276,9.25179
3,0.707,42.1,3000.0,17.3,2.89277,9.25177
4,0.707,35.4,3000.0,13.2,2.89280,9.25237
5,0.707,35.4,1730.0,13.2,2.89280,9.25240
6,0.707,35.4,548.0,17.3,2.89281,9.25214
7,0.707,35.4,1000.0,17.3,2.89282,9.25207
8,0.707,35.4,1000.0,13.2,2.89282,9.25246
9,0.707,42.1,3000.0,13.2,2.89282,9.25220


season_f {0.1: 2.89717, 0.25: 2.89435, 0.354: 2.89343, 0.5: 2.89286, 0.707: 2.89276, 1.0: 2.89326}
half_life {25.0: 2.89351, 35.4: 2.8928, 42.1: 2.89276, 50.0: 2.89286, 59.5: 2.89295, 70.7: 2.89309}
k_type {300.0: 2.89283, 548.0: 2.89276, 1000.0: 2.89276, 1730.0: 2.89276, 3000.0: 2.89277}
k_hand {10.0: 2.89339, 13.2: 2.8928, 17.3: 2.89276, 22.8: 2.8932, 30.0: 2.89406}


,season_f,half_life,k_type,k_hand,이득 ×1000,95% 하한,95% 상한
전체 1등,0.707,42.1,1000.0,17.3,0.4986,0.0236,1.0079
배수 1에서 1등,1.000,35.4,1000.0,17.3,0.0000,0.0000,0.0000


#### 선택: 2019 1등 그대로 (시즌 초 배수 0.707, 반감 42.1, k_type 1000, k_hand 17.3)
- 2019 24칸 log loss (실제 구종): model_09 2.89326 -> 2.89276 (−0.0005, 경기 단위 95% −0.0010 ~ −0.00002)
- 배수 1에서 다시 고르면 model_09 값 그대로 -> 이득은 전부 시즌 초 배수와 함께 움직인 몫
- 민감도: 배수 0.5–0.707 평평, 0.25 이하는 나빠짐 (0.1이면 +0.004) -> 구종 B4(0.25)보다 훨씬 약하게 줄임
- 반감 35.4 -> 42.1: 배수만 바꾼 값(2.89282)과 차이 0.00006 (잡음 수준) / 구종 model_06처럼 지난 시즌 꼬리를 줄이니 시즌 안 기록을 더 길게 보는 방향
  - 규칙대로 연결된 파라미터를 함께 고른 2019 1등을 씀 (35.4는 덜 극단적인 값이라 하기 어려움, 사용자 결정 2026-11-10)

In [16]:
LOC_PICK = best_of(res, GRID_SF)  # 2019 1등 (평평한 구간 규칙은 위 표를 보고 판단)
LOC_SEASON_F, LOC_HL, K_TYPE, K_HAND = LOC_PICK["season_f"], LOC_PICK["half_life"], LOC_PICK["k_type"], LOC_PICK["k_hand"]
print(f"위치: 시즌 초 배수 {LOC_SEASON_F}, 반감 {LOC_HL}, k_type {K_TYPE}, k_hand {K_HAND}")

위치: 시즌 초 배수 0.707, 반감 42.1, k_type 1000.0, k_hand 17.3


### 3-2. 진단 다시: 0-1-4 그룹별 (model_09 vs model_11, 실제 구종 기준)

- 0-1에서 어긋났던 시즌 첫 1–3등판이 고쳐졌는지, 다른 그룹이 나빠지지 않았는지 (2019 판단, 2020 확인)
- 이득 = model_09 − model_11 공별 24칸 log loss (양수 = 좋아짐)

In [17]:
app2 = m.drop_duplicates(["pitcher_code", "gameID"])[["pitcher_code", "gameID", "game_date", "GYEAR"]].sort_values(["game_date", "gameID"])
app2["app_no"] = app2.groupby(["pitcher_code", "GYEAR"]).cumcount() + 1  # 그 시즌 몇 번째 등판 (0-1과 같음)
_, NH2 = fc.lookup(fc.prior_table(fp, K2, ["x"]), fp, K2, ["x"])  # 투수×구종×손 전날까지 초구 수 (가중 없음)
rows = []
for yr, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]:
    i, yz = np.arange(len(q)), q["zone"].map(fc.ZONE_IDX).to_numpy()
    nll_ = lambda P: -np.log(np.clip(P[i, yz], 1e-12, None))
    P_new = loc_given_type(q, yr, q["pitch_type"].to_numpy(), LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F)[0]
    d = pd.DataFrame({"ll_13": fc.align(m09, fp, yr)["ll_zone_true"].to_numpy(), "ll_15": nll_(P_new),
                      "C_15":P_new[:, fc.ZONE_GROUP == "C"].sum(1), "C 실제": q["zone_group"].eq("C").to_numpy()})
    d["이득"] = d.ll_13 - d.ll_15
    d["등판"] = pd.cut(q[["pitcher_code", "gameID"]].merge(app2, how="left")["app_no"].to_numpy(), [1, 2, 4, 11, 999], right=False, labels=["1", "2–3", "4–10", "11+"])
    d["월"] = q["game_date"].dt.month.clip(4, 9).to_numpy()
    d["기록 수"] = pd.cut(NH2[fp["GYEAR"].eq(yr).to_numpy(), 0], [0, 1, 31, 101, 301, 1e9], right=False, labels=["0", "1–30", "31–100", "101–300", "301+"])
    for var in ["등판", "월", "기록 수"]:
        for v, g in d.groupby(var, observed=True):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "model_09 ll": g.ll_13.mean(), "model_11 ll": g.ll_15.mean(),
                         "이득 ×1000": g["이득"].mean() * 1000, "rC model_11": (g["C 실제"] - g.C_15).mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "model_09 ll": d.ll_13.mean(), "model_11 ll": d.ll_15.mean(),
                 "이득 ×1000": d["이득"].mean() * 1000, "rC model_11": (d["C 실제"] - d.C_15).mean()})
T = pd.DataFrame(rows).set_index(["변수", "값", "연도"])[["공", "model_09 ll", "model_11 ll", "이득 ×1000", "rC model_11"]].unstack("연도")
display(T.round(4))

공          model_09 ll         model_11 ll         이득 ×1000         rC model_11        
연도               2019     2020        2019    2020        2019    2020     2019    2020        2019    2020
변수   값                                                                                                     
기록 수 0           89.0     34.0      2.8857  2.9675      2.8858  2.9678  -0.0186 -0.2449     -0.0442  0.1191
     101–300   5672.0   5826.0      2.8927  2.8834      2.8920  2.8830   0.7052  0.4650     -0.0104 -0.0088
     1–30      1620.0    729.0      2.8820  2.9631      2.8812  2.9635   0.8137 -0.3187     -0.0218 -0.0001
     301+      2461.0   3671.0      2.9397  2.9283      2.9396  2.9278   0.0766  0.4592     -0.0291 -0.0082
     31–100    3616.0   2355.0      2.8677  2.8585      2.8673  2.8586   0.3332 -0.0786     -0.0238 -0.0107
등판   1          419.0    455.0      2.9560  2.9285      2.9526  2.9300   3.4423 -1.4690     -0.0708 -0.0346
     11+       8741.0   8106.0      2.8845  2.8801      2.8839  2.8799   0.6041  0.1726     -0.0155 -0.0124
     2–3        925.0    934.0      2.9105  2.9438      2.9103  2.9418   0.2225  2.0196     -0.0104 -0.0064
     4–10      3373.0   3120.0      2.9033  2.9210      2.9034  2.9205  -0.0648  0.4333     -0.0243  0.0063
월    4         2568.0      NaN      2.9247     NaN      2.9231     NaN   1.5692     NaN     -0.0254     NaN
     5         2403.0   2264.0      2.9057  2.9259      2.9061  2.9246  -0.4221  1.2776     -0.0231 -0.0119
     6         2493.0   2379.0      2.9033  2.9201      2.9029  2.9186   0.4721  1.4859     -0.0245  0.0024
     7         1791.0   2233.0      2.8509  2.8995      2.8499  2.8996   1.0289 -0.0691      0.0090 -0.0106
     8         2362.0   1985.0      2.8969  2.8763      2.8968  2.8770   0.0781 -0.6656     -0.0207 -0.0133
     9         1841.0   3754.0      2.8561  2.8733      2.8558  2.8735   0.2665 -0.2619     -0.0226 -0.0083
전체            13458.0  12615.0      2.8933  2.8967      2.8928  2.8963   0.4986  0.3146     -0.0190 -0.0081

#### 🔍 3-2 결과: 전체로는 조금 좋아졌지만, 노렸던 시즌 첫 등판은 두 해 방향이 다름
- 전체 이득 ×1000 (실제 구종): 2019 +0.50, 2020 +0.31
- 그 시즌 첫 등판: 2019 +3.4, 2020 **−1.5** / 2–3번째: 2019 +0.2, 2020 +2.0 -> 0-1에서 본 첫 등판 어긋남을 두 해 일관되게 고치지는 못함
  - 첫 등판 rC는 그대로 (2019 −0.071 -> −0.071, 2020 −0.036 -> −0.035) -> 0-1의 C 과다 예측은 지난 시즌 기록을 너무 믿어서 생긴 게 아님
- 월: 2019 4월 +1.6, 2020 첫 달(5월) +1.3 · 6월 +1.5 -> 시즌 앞쪽 달에 이득이 몰린 점은 두 해 같음
- 기록 수: 101–300구가 두 해 모두 +0.5·+0.5, 1–30구는 2019 +0.8 / 2020 −0.3


## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_09과 같음, 위치만 3-1에서 고른 값 (시즌 초 배수 포함)

In [18]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [19]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 위치와 무관한 열(구종)은 model_09과 같아야 함
chk = pp.merge(m09, on=fc.KEYS, suffixes=("", "_09"))
same = ["ll_type"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_09"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_09과 다른 열: {bad}"
print("구종 = model_09 (재현 확인), 위치만 바뀜")

(26073, 48)
구종 = model_09 (재현 확인), 위치만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_09과 나란히 / 경기 단위 부트스트랩 (model_11 − model_09, 2020)
- 2020은 확인용 -> 2020을 보고 파라미터 다시 고르지 않음

In [20]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m09_sum = eval_years(m09)
display(pd.concat({MODEL: summary, "model_09": m09_sum, "차이 (model_09 대비)": summary - m09_sum}, axis=1).round(4))

model_11          model_09          차이 (model_09 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9491   2.9453   2.9493   2.9454          -0.0002 -0.0001
24칸 top1              0.1303   0.1331   0.1300   0.1337           0.0002 -0.0006
24칸 top3              0.2903   0.2920   0.2888   0.2908           0.0015  0.0012
H/S/C logloss         1.0747   1.0765   1.0748   1.0767          -0.0001 -0.0002
H/S/C 정확도             0.3953   0.3898   0.3951   0.3896           0.0002  0.0002
24칸 logloss (실제 구종)   2.8928   2.8963   2.8933   2.8967          -0.0005 -0.0003
구종×칸 logloss          3.9541   3.9518   3.9546   3.9521          -0.0005 -0.0003
좌표 평균거리 cm           32.6727  31.8253  32.6837  31.8305          -0.0110 -0.0052
좌표 NLL                9.3168   9.2661   9.3171   9.2660          -0.0004  0.0001

In [21]:
display(fc.bootstrap_diff(fc.pitch_scores(m09, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0000,0.0000,0.0000,차이 불확실
ll_zone,-0.0001,-0.0005,0.0003,차이 불확실
ll_joint,-0.0003,-0.0009,0.0003,차이 불확실
nll_coord,0.0001,-0.0004,0.0006,차이 불확실
dist,-0.0052,-0.0110,0.0004,차이 불확실


### 🥇1등🥇
#### 2020 24칸 log loss 실제 구종 2.8967 -> 2.8963 (−0.0003), 실전 2.9454 -> 2.9453, 나빠진 점수 없음 -> 채택
- 구종은 model_09과 같음 (재현 확인) -> 실전 변화 = 순수한 위치 효과
- 부트스트랩 (2020, model_11 − model_09): 6개 점수 모두 차이 불확실, 나빠짐 없음 (24칸 −0.0001, 동시 −0.0003, 거리 −0.005cm)
- 24칸 top3 +0.0015 / +0.0012 (2019 / 2020)
- 이득은 시즌 앞쪽 달에 몰림 (두 해 같음), 그런데 0-1에서 본 시즌 첫 등판 C 과다 예측(rC −0.07)은 그대로 (3-2)
- 크기는 model_07(두 해 −0.0003, 채택)과 비슷: 2019 1등 + 95% 구간 0 위 + 2020 같은 방향·나빠짐 없음
- **채택 -> 다음 비교 출발점** (사용자 결정 2026-11-10)

## 6. 저장

- `results/model_11/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [22]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_11 ['per_pitch.parquet', 'summary.csv']
